# 07 Final model export
**Goal:** train the shipped model, save it, and prove that the serving function matches the training features.

**Held-out demo:** the model is trained only on data **before the last walk-forward test fold** (up to 2017-12-02 23:00), so the app's demo days (Dec 3 - Dec 30) were never seen in training. Cost: about 8% less training data.

**Needs:** `src/forecast_lib.py`. **Outputs:** `models/lgbm_ratio_noweather.joblib`, `data/demo/demo_history.csv`.

In [1]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, joblib
from pathlib import Path
import forecast_lib as fl

h = pd.read_csv("../data/processed/hourly.csv", index_col=0, parse_dates=True)
F = fl.build_features(h)
tscv = fl.make_tscv()

In [2]:
# Train on everything before the last test fold, save model and demo data
tr, te = list(tscv.split(F))[-1]
te0 = te[0]
print("train until:", F.index[te0 - 1], "| demo days start:", F.index[te0])

final = fl.train_final(F.iloc[:te0])
Path("../models").mkdir(exist_ok=True)
joblib.dump({"model": final, "features": fl.REL_NOWX,
             "trained_until": str(F.index[te0 - 1])},
            "../models/lgbm_ratio_noweather.joblib")

Path("../data/demo").mkdir(parents=True, exist_ok=True)
h[["total"]].loc["2017-11-25":].to_csv("../data/demo/demo_history.csv")

train until: 2017-12-02 23:00:00 | demo days start: 2017-12-03 00:00:00


In [3]:
# Parity test: reload the SAVED model; predict_next_day must equal the batch feature pipeline.
# (Dec 30 is held out of training, but this checks code agreement, not accuracy.)
bundle = joblib.load("../models/lgbm_ratio_noweather.joblib")
model = bundle["model"]

day = pd.Timestamp("2017-12-30")
hist = h["total"].loc[: day - pd.Timedelta(hours=1)]
out = fl.predict_next_day(model, hist)

X = F.loc[day: day + pd.Timedelta(hours=23), fl.REL_NOWX]
ref = model.predict(X) * F.loc[X.index, "roll_mean_7d"].to_numpy()
print("trained_until:", bundle["trained_until"])
print("max abs diff vs batch features:", np.abs(out.to_numpy() - ref).max())
print(out.round(0).head(3))

trained_until: 2017-12-02 23:00:00
max abs diff vs batch features: 0.0
2017-12-30 00:00:00    60377.0
2017-12-30 01:00:00    53462.0
2017-12-30 02:00:00    49203.0
Freq: h, Name: forecast, dtype: float64


## Conclusion
- Final model: LightGBM on load ÷ past-week mean, calendar + three ratio lags, no weather, trained through 2017-12-02.
- The serving function needs 8 days (192 h) of past hourly load ending at 23:00 and never asks for future values.
- Parity test: the difference between the serving function and the batch pipeline must be 0.0.